In [5]:
import numpy as np 
import pandas as pd 
df=pd.read_csv("cpcb_dly.csv")

In [3]:
df_clean = df.copy()

# Standardize column names
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(df_clean.columns.tolist())

['stn_code', 'sampling_date', 'state', 'city_town_village_area', 'location_of_monitoring_station', 'agency', 'type_of_location', 'so2', 'no2', 'rspm_pm10', 'pm_2_5']


In [6]:
text_cols = df_clean.select_dtypes(
    include=["object", "str"]
).columns

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()
    df_clean[col] = df_clean[col].replace("", np.nan)

print("Text columns cleaned.")

Text columns cleaned.


In [7]:
df_clean["sampling_date"] = pd.to_datetime(
    df_clean["sampling_date"],
    format="%d-%m-%y",
    errors="coerce"
)

pollutant_cols = ["so2", "no2", "rspm_pm10", "pm_2_5"]

for col in pollutant_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

print(df_clean.dtypes)

stn_code                                   int64
sampling_date                     datetime64[us]
state                                        str
city_town_village_area                       str
location_of_monitoring_station               str
agency                                       str
type_of_location                             str
so2                                      float64
no2                                      float64
rspm_pm10                                float64
pm_2_5                                   float64
dtype: object


In [8]:
print("Exact duplicates:", df_clean.duplicated().sum())

Exact duplicates: 0


In [9]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_clean.isna().sum(),
    "Missing_Percentage": (
        df_clean.isna().mean() * 100
    ).round(2)
})

missing_summary.sort_values(
    "Missing_Percentage",
    ascending=False
)

,Missing_Count,Missing_Percentage
pm_2_5,1848,80.28
no2,25,1.09
so2,21,0.91
rspm_pm10,7,0.30
state,0,0.00
stn_code,0,0.00
sampling_date,0,0.00
type_of_location,0,0.00
agency,0,0.00
location_of_monitoring_station,0,0.00


In [10]:
print("Final rows:", len(df_clean))
print("Final columns:", len(df_clean.columns))
print("Exact duplicates:", df_clean.duplicated().sum())
print("Invalid dates:", df_clean["sampling_date"].isna().sum())

display(
    pd.DataFrame({
        "Data_Type": df_clean.dtypes.astype(str),
        "Missing_Count": df_clean.isna().sum(),
        "Missing_Percentage": (
            df_clean.isna().mean() * 100
        ).round(2)
    })
)

Final rows: 2302
Final columns: 11
Exact duplicates: 0
Invalid dates: 0


,Data_Type,Missing_Count,Missing_Percentage
stn_code,int64,0,0.00
sampling_date,datetime64[us],0,0.00
state,str,0,0.00
city_town_village_area,str,0,0.00
location_of_monitoring_station,str,0,0.00
agency,str,0,0.00
type_of_location,str,0,0.00
so2,float64,21,0.91
no2,float64,25,1.09
rspm_pm10,float64,7,0.30


In [11]:
df_clean.to_csv("tamil_nadu_air_quality_cleaned.csv", index=False)

### Data Cleaning Summary

The original Tamil Nadu air quality dataset contained 2,302 observations and 11 columns covering 2015. Column names and text values were standardized, the sampling date was converted to datetime format, and pollutant columns were validated as numeric.

The final dataset retained all 2,302 records and 11 columns. No exact duplicate rows or invalid dates were found. Negative pollutant concentrations were not detected in the initial validation.

Missing values were retained rather than replaced with zero. PM2.5 had 80.28% missing data, while SO2, NO2 and RSPM/PM10 had less than 1.1% missing data each. These limitations will be considered when interpreting the analysis.